# Phase 3 — QLoRA Fine-Tuning
### Train Qwen 1.5B on Spider SQL Dataset

This notebook:
1. Loads the formatted Spider dataset
2. Configures QLoRA (4-bit quantization + LoRA adapters)
3. Trains using SFTTrainer with W&B logging
4. Saves the LoRA adapters to disk


In [1]:
import os
import torch
import wandb
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print("All imports successful!")
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name      : {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Total VRAM    : {vram:.1f} GB")


All imports successful!
GPU available : True
GPU name      : NVIDIA GeForce RTX 5060 Laptop GPU
Total VRAM    : 8.0 GB


## Step 1 — Training Configuration

In [2]:
# ── All hyperparameters in one place ─────────────────────────────
CONFIG = {
    # Model
    "model_name"        : "Qwen/Qwen1.5-1.8B-Chat",
    "data_dir"          : "./data/spider_formatted",

    # LoRA hyperparameters
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "lora_target"       : ["q_proj", "v_proj", "k_proj", "o_proj"],

    # Training hyperparameters
    "num_epochs"        : 5,       # ← changed
    "batch_size"        : 8,       # ← changed
    "grad_accum_steps"  : 2,       # ← changed
    "learning_rate"     : 2e-4,
    "max_seq_length"    : 512,
    "warmup_ratio"      : 0.05,
    "lr_scheduler"      : "cosine",
    "weight_decay"      : 0.01,

    # Output
    "output_dir"        : "./outputs/qwen-sql-qlora",
    "save_steps"        : 100,
    "eval_steps"        : 100,
    "logging_steps"     : 10,

    # W&B
    "wandb_project"     : "sql-finetuning",
    "wandb_run_name"    : "qwen1.5-spider-qlora-r16-v2",  # ← v2 for new run
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("Configuration ready!")
print(f"\nKey hyperparameters:")
print(f"  LoRA rank       : {CONFIG['lora_r']}")
print(f"  LoRA alpha      : {CONFIG['lora_alpha']}")
print(f"  Learning rate   : {CONFIG['learning_rate']}")
print(f"  Epochs          : {CONFIG['num_epochs']}")
print(f"  Batch size      : {CONFIG['batch_size']} x {CONFIG['grad_accum_steps']} = {CONFIG['batch_size'] * CONFIG['grad_accum_steps']} effective")


Configuration ready!

Key hyperparameters:
  LoRA rank       : 16
  LoRA alpha      : 32
  Learning rate   : 0.0002
  Epochs          : 5
  Batch size      : 8 x 2 = 16 effective


## Step 2 — Initialize Weights & Biases

In [3]:
# Initialize W&B run
wandb.init(
    project = CONFIG["wandb_project"],
    name    = CONFIG["wandb_run_name"],
    config  = CONFIG,
)

print(f"W&B run initialized!")
print(f"Project  : {CONFIG['wandb_project']}")
print(f"Run name : {CONFIG['wandb_run_name']}")
print(f"Track at : https://wandb.ai")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tejaa\_netrc.
wandb: Currently logged in as: arunteja962 (arunteja962-aispry) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


W&B run initialized!
Project  : sql-finetuning
Run name : qwen1.5-spider-qlora-r16-v2
Track at : https://wandb.ai


## Step 3 — Load Dataset

In [10]:
dataset = load_from_disk(CONFIG["data_dir"])

# 500 samples for fast training
train_data = dataset["train"].select(range(3500))
val_data   = dataset["validation"].select(range(200))

print(f"Train samples      : {len(train_data)}")
print(f"Validation samples : {len(val_data)}")

Train samples      : 3500
Validation samples : 200


## Step 4 — Configure 4-bit Quantization (QLoRA)

In [11]:
# BitsAndBytes 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,          # load model in 4-bit
    bnb_4bit_quant_type       = "nf4",         # NormalFloat4 quantization
    bnb_4bit_compute_dtype    = torch.float16, # compute in fp16
    bnb_4bit_use_double_quant = True,          # nested quantization
)

print("4-bit quantization config ready!")
print("  Type             : NF4 (NormalFloat4)")
print("  Compute dtype    : float16")
print("  Double quant     : True")
print("\nThis reduces model VRAM from ~7GB to ~1.5GB!")


4-bit quantization config ready!
  Type             : NF4 (NormalFloat4)
  Compute dtype    : float16
  Double quant     : True

This reduces model VRAM from ~7GB to ~1.5GB!


## Step 5 — Load Model with Quantization

In [12]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"
print("Tokenizer loaded!")

# Load model in 4-bit
print("\nLoading model in 4-bit (from cache, should be fast)...")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config = bnb_config,
    device_map          = {"": 0},    # ← changed from "cuda" to {"": 0}
    trust_remote_code   = True,
)
model.config.use_cache = False

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nModel loaded in 4-bit!")
print(f"VRAM used : {vram_used:.2f} GB (vs ~7GB in fp16)")

Loading tokenizer...
Tokenizer loaded!

Loading model in 4-bit (from cache, should be fast)...

Model loaded in 4-bit!
VRAM used : 4.29 GB (vs ~7GB in fp16)


## Step 6 — Apply LoRA Adapters

In [13]:
# LoRA configuration
lora_config = LoraConfig(
    r              = CONFIG["lora_r"],
    lora_alpha     = CONFIG["lora_alpha"],
    lora_dropout   = CONFIG["lora_dropout"],
    target_modules = CONFIG["lora_target"],
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nVRAM after LoRA : {vram_used:.2f} GB")
print("Only the small LoRA adapters will be trained!")


trainable params: 6,291,456 || all params: 1,843,120,128 || trainable%: 0.3413

VRAM after LoRA : 4.31 GB
Only the small LoRA adapters will be trained!


## Step 7 — Training Arguments

In [14]:
training_args = SFTConfig(
    # Output
    output_dir                  = CONFIG["output_dir"],

    # Training schedule
    num_train_epochs            = CONFIG["num_epochs"],
    per_device_train_batch_size = CONFIG["batch_size"],
    per_device_eval_batch_size  = CONFIG["batch_size"],
    gradient_accumulation_steps = CONFIG["grad_accum_steps"],

    # Optimizer
    learning_rate               = CONFIG["learning_rate"],
    weight_decay                = CONFIG["weight_decay"],
    lr_scheduler_type           = CONFIG["lr_scheduler"],
    warmup_ratio                = CONFIG["warmup_ratio"],

    # Precision
    fp16                        = True,
    bf16                        = False,

    # Logging & saving
    logging_steps               = CONFIG["logging_steps"],
    save_steps                  = CONFIG["save_steps"],
    eval_steps                  = CONFIG["eval_steps"],
    evaluation_strategy         = "steps",
    save_strategy               = "steps",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",

    # W&B
    report_to                   = "wandb",
    run_name                    = CONFIG["wandb_run_name"],

    # Sequence length
    max_seq_length              = CONFIG["max_seq_length"],
    dataset_text_field          = "text",

    # Misc
    dataloader_pin_memory       = False,
    group_by_length             = True,
)

print("Training arguments configured!")


Training arguments configured!


## Step 8 — Initialize Trainer

In [15]:
trainer = SFTTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_data,
    eval_dataset    = val_data,
    tokenizer       = tokenizer,
)

print("Trainer initialized!")
print(f"\nTraining summary:")
print(f"  Total samples     : {len(train_data)}")
print(f"  Epochs            : {CONFIG['num_epochs']}")
print(f"  Effective batch   : {CONFIG['batch_size'] * CONFIG['grad_accum_steps']}")
total_steps = (len(train_data) // (CONFIG['batch_size'] * CONFIG['grad_accum_steps'])) * CONFIG['num_epochs']
print(f"  Total steps       : ~{total_steps}")
print(f"  Saving to         : {CONFIG['output_dir']}")
print(f"\nEstimated training time on RTX 5060: ~45-90 minutes")


Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Trainer initialized!

Training summary:
  Total samples     : 3500
  Epochs            : 5
  Effective batch   : 16
  Total steps       : ~1090
  Saving to         : ./outputs/qwen-sql-qlora

Estimated training time on RTX 5060: ~45-90 minutes


## Step 9 — Start Training! 🔥

In [16]:
print("Starting training...")
print("Watch your W&B dashboard for live loss curves!")
print("=" * 50)

# Train!
trainer.train()

print("=" * 50)
print("Training complete!") 


Starting training...
Watch your W&B dashboard for live loss curves!


  0%|          | 0/1095 [00:00<?, ?it/s]

{'loss': 4.2296, 'grad_norm': 4.953376293182373, 'learning_rate': 3.6363636363636364e-05, 'epoch': 0.05}
{'loss': 3.4391, 'grad_norm': 2.5650863647460938, 'learning_rate': 7.272727272727273e-05, 'epoch': 0.09}
{'loss': 2.2755, 'grad_norm': 3.4749324321746826, 'learning_rate': 0.00010909090909090909, 'epoch': 0.14}
{'loss': 1.0303, 'grad_norm': 0.8550471663475037, 'learning_rate': 0.00014545454545454546, 'epoch': 0.18}
{'loss': 0.5763, 'grad_norm': 0.5595968961715698, 'learning_rate': 0.00018181818181818183, 'epoch': 0.23}
{'loss': 0.7781, 'grad_norm': 0.42457252740859985, 'learning_rate': 0.0001999885939617498, 'epoch': 0.27}
{'loss': 0.662, 'grad_norm': 0.3741082549095154, 'learning_rate': 0.00019989736126687963, 'epoch': 0.32}
{'loss': 0.654, 'grad_norm': 0.4329968988895416, 'learning_rate': 0.00019971497912068013, 'epoch': 0.37}
{'loss': 0.5829, 'grad_norm': 0.41748589277267456, 'learning_rate': 0.00019944161393427922, 'epoch': 0.41}
{'loss': 0.4626, 'grad_norm': 0.4120979607105255,

  0%|          | 0/25 [00:00<?, ?it/s]

{'eval_loss': 0.7266244292259216, 'eval_runtime': 28.0713, 'eval_samples_per_second': 7.125, 'eval_steps_per_second': 0.891, 'epoch': 0.46}
{'loss': 0.6904, 'grad_norm': 0.39221352338790894, 'learning_rate': 0.00019862301493654108, 'epoch': 0.5}
{'loss': 0.5811, 'grad_norm': 0.40253663063049316, 'learning_rate': 0.00019807852804032305, 'epoch': 0.55}
{'loss': 0.5925, 'grad_norm': 0.42371007800102234, 'learning_rate': 0.0001974445512526336, 'epoch': 0.59}
{'loss': 0.5178, 'grad_norm': 0.47058650851249695, 'learning_rate': 0.00019672166303356028, 'epoch': 0.64}
{'loss': 0.3955, 'grad_norm': 0.4551025331020355, 'learning_rate': 0.00019591052296873888, 'epoch': 0.68}
{'loss': 0.6218, 'grad_norm': 0.48314180970191956, 'learning_rate': 0.00019501187116752693, 'epoch': 0.73}
{'loss': 0.529, 'grad_norm': 0.43141573667526245, 'learning_rate': 0.00019402652758770475, 'epoch': 0.78}
{'loss': 0.5524, 'grad_norm': 0.41670146584510803, 'learning_rate': 0.00019295539128732093, 'epoch': 0.82}
{'loss':

...

{'eval_loss': 0.8564505577087402, 'eval_runtime': 25.748, 'eval_samples_per_second': 7.768, 'eval_steps_per_second': 0.971, 'epoch': 3.65}
{'loss': 0.2608, 'grad_norm': 0.540522575378418, 'learning_rate': 3.482579126965878e-05, 'epoch': 3.7}
{'loss': 0.3117, 'grad_norm': 0.7201213240623474, 'learning_rate': 3.2564804286991135e-05, 'epoch': 3.74}
{'loss': 0.2882, 'grad_norm': 0.6290880441665649, 'learning_rate': 3.036534726879473e-05, 'epoch': 3.79}
{'loss': 0.3124, 'grad_norm': 0.6027551889419556, 'learning_rate': 2.8229427067882164e-05, 'epoch': 3.84}
{'loss': 0.266, 'grad_norm': 0.5883160829544067, 'learning_rate': 2.6158992564103058e-05, 'epoch': 3.88}
{'loss': 0.2259, 'grad_norm': 0.5073256492614746, 'learning_rate': 2.415593288612541e-05, 'epoch': 3.93}
{'loss': 0.2965, 'grad_norm': 0.5638762712478638, 'learning_rate': 2.2222075687736187e-05, 'epoch': 3.97}
{'loss': 0.249, 'grad_norm': 0.5754882097244263, 'learning_rate': 2.035918548023329e-05, 'epoch': 4.02}
{'loss': 0.278, 'grad

  0%|          | 0/25 [00:00<?, ?it/s]

{'eval_loss': 0.8817718625068665, 'eval_runtime': 24.8821, 'eval_samples_per_second': 8.038, 'eval_steps_per_second': 1.005, 'epoch': 4.11}
{'loss': 0.2712, 'grad_norm': 0.5908372402191162, 'learning_rate': 1.5212981383783154e-05, 'epoch': 4.16}
{'loss': 0.2327, 'grad_norm': 0.5607429146766663, 'learning_rate': 1.3650286303778714e-05, 'epoch': 4.2}
{'loss': 0.2269, 'grad_norm': 0.5449668765068054, 'learning_rate': 1.2166379381199423e-05, 'epoch': 4.25}
{'loss': 0.271, 'grad_norm': 0.5827332139015198, 'learning_rate': 1.0762614578753572e-05, 'epoch': 4.29}
{'loss': 0.2667, 'grad_norm': 0.6079262495040894, 'learning_rate': 9.440272734993072e-06, 'epoch': 4.34}
{'loss': 0.2714, 'grad_norm': 0.5830894112586975, 'learning_rate': 8.200560395636414e-06, 'epoch': 4.38}
{'loss': 0.2348, 'grad_norm': 0.5738392472267151, 'learning_rate': 7.0446087126790575e-06, 'epoch': 4.43}
{'loss': 0.2218, 'grad_norm': 0.7016064524650574, 'learning_rate': 5.973472412295255e-06, 'epoch': 4.47}
{'loss': 0.2714, 

  0%|          | 0/25 [00:00<?, ?it/s]

{'eval_loss': 0.8881950378417969, 'eval_runtime': 26.2185, 'eval_samples_per_second': 7.628, 'eval_steps_per_second': 0.954, 'epoch': 4.57}
{'loss': 0.2738, 'grad_norm': 0.559293270111084, 'learning_rate': 3.2783369664397436e-06, 'epoch': 4.61}
{'loss': 0.24, 'grad_norm': 0.5854061245918274, 'learning_rate': 2.55544874736644e-06, 'epoch': 4.66}
{'loss': 0.2321, 'grad_norm': 0.5386561155319214, 'learning_rate': 1.921471959676957e-06, 'epoch': 4.7}
{'loss': 0.272, 'grad_norm': 0.5670698881149292, 'learning_rate': 1.3769850634589354e-06, 'epoch': 4.75}
{'loss': 0.2541, 'grad_norm': 0.6424442529678345, 'learning_rate': 9.224848654469931e-07, 'epoch': 4.79}
{'loss': 0.2663, 'grad_norm': 0.5241991877555847, 'learning_rate': 5.58386065720784e-07, 'epoch': 4.84}
{'loss': 0.2266, 'grad_norm': 0.6146989464759827, 'learning_rate': 2.850208793198861e-07, 'epoch': 4.89}
{'loss': 0.2221, 'grad_norm': 0.5615983605384827, 'learning_rate': 1.0263873312040818e-07, 'epoch': 4.93}
{'loss': 0.2657, 'grad_n

## Step 10 — Save LoRA Adapters

In [17]:
# Save the LoRA adapters (NOT the full model - much smaller!)
adapter_path = CONFIG["output_dir"] + "/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapters saved to: {adapter_path}")

# Check adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter size: {total_size / 1024**2:.1f} MB (vs ~3.5GB for full model!)")

# Finish W&B run
wandb.finish()
print("\nW&B run finished!")
print("\nPhase 3 Complete!")
print("LoRA adapters saved and ready for evaluation.")
print("Next up: Phase 4 - Post Fine-Tuning Evaluation!")


LoRA adapters saved to: ./outputs/qwen-sql-qlora/final_adapter
Adapter size: 35.0 MB (vs ~3.5GB for full model!)


eval/loss,▁▃▂▂▄▅▆▇██
eval/runtime,▅▇███▂▃▂▁▃
eval/samples_per_second,▄▂▁▁▁▇▆▇█▆
eval/steps_per_second,▄▂▁▁▁▇▆▇█▆
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/grad_norm,█▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁
train/learning_rate,▅▇█████████▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▂▂▂▂▁▁▁▁▁▁▁
train/loss,█▂▃▂▂▂▂▂▂▂▂▁▂▂▂▁▁▁▁▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.8882
eval/runtime,26.2185



W&B run finished!

Phase 3 Complete!
LoRA adapters saved and ready for evaluation.
Next up: Phase 4 - Post Fine-Tuning Evaluation!
